### **NOTEBOOK 03: SAGEMAKER PIPELINE WITH POST-PIPELINE MLFLOW APP LOGGING**
#### **TEAM 04 STUDENT 3 - TING ENG KIAT (S403)**
#### **Jupyter Notebook Version: 26 August 2026**
#### **Model B - Multilayer Perceptron (MLP) for CI/CD Workflow with Retraining Trigger**

In [2]:
# INSTALL/UPGRADE SAGEMAKER, BOTO3, BOTOCORE, MLFLOW, SAGEMAKER-MLFLOW PACKAGES
# After running, restart the kernel before continuing if packages were upgraded.
%pip install --upgrade "sagemaker>=2,<3" boto3 botocore mlflow sagemaker-mlflow


Note: you may need to restart the kernel to use updated packages.


### **STEP 01: CONFIGURATION**

In [1]:
import boto3
import sagemaker
import json
import os
import time
from pathlib import Path

# ----------------------------
# AWS / SageMaker setup
# ----------------------------
session = sagemaker.Session()
role    = sagemaker.get_execution_role()
region  = boto3.Session().region_name

# Use the ITI113 course bucket and team prefix.
# This matches the team execution role S3 policy, e.g.:
# s3://nyp-26s1-iti113/iti113/team40/
BUCKET  = "nyp-26s1-iti113"

# Change these for the current student/profile.
TEAM_ID = "team04"
STUDENT_ID = "s403"

COURSE = "ITI113"
SEMESTER = "26S1"
PROJECT_NAME = "credit-card-fraud-detection"

PREFIX  = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"

# Instance types.
# If ml.m5.large quota is 0, change these to an approved available training/processing type.
# For ITI113, keep Studio spaces on ml.t3.medium and use SageMaker jobs for training/processing.
PROCESSING_INSTANCE_TYPE = "ml.m5.large"
TRAINING_INSTANCE_TYPE   = "ml.m5.large"

# ----------------------------
# SageMaker Serverless MLflow App setup
# ----------------------------
# Preferred:
# 1. Team-level config copied/shared from Notebook 01A:
#       mlflow_app_config_team40.json
# 2. Student-specific config from Notebook 01A:
#       mlflow_app_config_team40_s4002.json
# 3. Any local config matching this team:
#       mlflow_app_config_team40_*.json
#
# Important:
# Do not use another team's MLflow ARN. With team-level IAM
# restriction, wrong-team access should fail with 403.
# ----------------------------

TEAM_CONFIG_FILE = Path(f"mlflow_app_config_{TEAM_ID}.json")
STUDENT_CONFIG_FILE = Path(f"mlflow_app_config_{TEAM_ID}_{STUDENT_ID}.json")

config_candidates = [
    TEAM_CONFIG_FILE,
    STUDENT_CONFIG_FILE,
    *sorted(Path(".").glob(f"mlflow_app_config_{TEAM_ID}_*.json")),
]

MLFLOW_APP_ARN = None
MLFLOW_EXPERIMENT_NAME = f"{COURSE}/{TEAM_ID}/Experiment-01"
mlflow_config = {}
config_used = None

for config_file in config_candidates:
    if config_file.exists():
        mlflow_config = json.loads(config_file.read_text(encoding="utf-8"))
        MLFLOW_APP_ARN = (
            mlflow_config.get("MLFLOW_APP_ARN")
            or mlflow_config.get("mlflow_app_arn")
            or mlflow_config.get("arn")
        )
        MLFLOW_EXPERIMENT_NAME = (
            mlflow_config.get("EXPERIMENT_NAME")
            or mlflow_config.get("experiment_name")
            or MLFLOW_EXPERIMENT_NAME
        )
        MODEL_DEVELOPER_ID = mlflow_config.get("STUDENT_ID") or mlflow_config.get("student_id")
        config_used = config_file
        break

# Fallback for classroom testing only.
# Update this to your team's MLflow App ARN from Notebook 01A if the config file
# is not available in this Studio workspace.
DEFAULT_MLFLOW_APP_ARN = (
    "arn:aws:sagemaker:ap-southeast-1:044528205969:"
    "mlflow-app/app-L5IMA5YSBDTY"
)

if MLFLOW_APP_ARN is None:
    MLFLOW_APP_ARN = DEFAULT_MLFLOW_APP_ARN
    print(
        "[WARNING] No local MLflow config file found. "
        "Using DEFAULT_MLFLOW_APP_ARN. Make sure this ARN belongs to your own team."
    )
else:
    print(f"Loaded MLflow App config from {config_used}")

# Validate config team, if present.
config_team_id = mlflow_config.get("TEAM_ID") or mlflow_config.get("team_id")
if config_team_id and config_team_id != TEAM_ID:
    raise ValueError(
        f"Config file team mismatch: config TEAM_ID={config_team_id}, notebook TEAM_ID={TEAM_ID}. "
        "Do not use another team's MLflow config."
    )

# ----------------------------
# Safety check for team-level MLflow restriction
# ----------------------------
# The selected MLflow App must have ResourceTag/TeamId = TEAM_ID.
# If a student accidentally uses another team's ARN, this should either:
# - fail with AccessDenied / 403 due to IAM restriction, or
# - fail this explicit validation before logging.
# ----------------------------

sm_for_mlflow = boto3.client("sagemaker", region_name=region)

try:
    tag_response = sm_for_mlflow.list_tags(ResourceArn=MLFLOW_APP_ARN)
    mlflow_app_tags = {t["Key"]: t["Value"] for t in tag_response.get("Tags", [])}

    print("MLflow App tags:")
    for k, v in mlflow_app_tags.items():
        print(f"  {k}: {v}")

    app_team_id = mlflow_app_tags.get("TeamId")
    if app_team_id != TEAM_ID:
        raise PermissionError(
            f"MLflow App TeamId tag mismatch. App TeamId={app_team_id}, notebook TEAM_ID={TEAM_ID}. "
            "Do not log to another team's MLflow App."
        )

    print(f"[OK] MLflow App tag TeamId={app_team_id} matches notebook TEAM_ID={TEAM_ID}")

except Exception as e:
    print("\n[ERROR] Could not validate MLflow App team tag.")
    print("This usually means one of the following:")
    print("1. The MLflow App ARN belongs to another team and IAM correctly blocked access.")
    print("2. The MLflow App is missing the TeamId tag.")
    print("3. The current role lacks permission to list tags for this MLflow App.")
    print(type(e).__name__, e)
    raise

# Optional: store for downstream cells and subprocesses.
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_APP_ARN
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME


# ----------------------------
# MLflow App UI link helpers
# ----------------------------
# MLflow may print generic links such as:
# https://mlflow.sagemaker.ap-southeast-1.app.aws/#/...
# Those links are not presigned and may show a permission/session error.
# Use these helpers to generate a fresh presigned SageMaker MLflow App URL
# and append the experiment/run fragment.

def create_mlflow_app_presigned_url(fragment: str = "") -> str:
    sm_for_mlflow = boto3.client("sagemaker", region_name=region)
    response = sm_for_mlflow.create_presigned_mlflow_app_url(
        Arn=MLFLOW_APP_ARN
    )

    base_url = response.get("AuthorizedUrl") or response.get("Url")

    if not base_url:
        raise RuntimeError(
            "create_presigned_mlflow_app_url did not return AuthorizedUrl or Url. "
            f"Response: {response}"
        )

    # Remove any existing fragment before appending our own MLflow UI route.
    base_url = base_url.split("#", 1)[0]

    if fragment:
        return base_url + "#" + fragment.lstrip("#")

    return base_url


def print_mlflow_presigned_links(experiment_id=None, run_id=None):
    if experiment_id is not None:
        experiment_url = create_mlflow_app_presigned_url(
            f"/experiments/{experiment_id}"
        )
        print("Presigned MLflow experiment URL:")
        print(experiment_url)

    if experiment_id is not None and run_id is not None:
        run_url = create_mlflow_app_presigned_url(
            f"/experiments/{experiment_id}/runs/{run_id}"
        )
        print("\nPresigned MLflow run URL:")
        print(run_url)

PIPELINE_NAME       = f"iti113-{TEAM_ID}-credit-card-fraud-detection"
MODEL_PACKAGE_GROUP = f"{TEAM_ID}-credit-card-fraud-detection"
ENDPOINT_NAME       = f"iti113-{TEAM_ID}-credit-card-fraud-detection-b"
QUALITY_GATE_AUC    = 0.35

RAW_DATA_URI  = f"s3://{BUCKET}/{PREFIX}/raw/fraudTest_cicd.csv"
PIPELINE_ROOT = f"s3://{BUCKET}/{PREFIX}/pipeline"

# Store the pipeline source files in S3 first, then download them into a clean local folder.
SCRIPTS_S3_PREFIX = f"{PREFIX}/pipeline_src"
SCRIPTS_S3_URI    = f"s3://{BUCKET}/{SCRIPTS_S3_PREFIX}"

LOCAL_PIPELINE_SRC = "pipeline_src"


print(f"Pipeline                : {PIPELINE_NAME}")
print(f"Bucket                  : {BUCKET}")
print(f"Team prefix             : {PREFIX}")
print(f"Semester                : {SEMESTER}")
print(f"Region                  : {region}")
print(f"SageMaker role          : {role}")
print(f"MLflow App ARN          : {MLFLOW_APP_ARN}")
print(f"MLflow experiment       : {MLFLOW_EXPERIMENT_NAME}")
print(f"Pipeline source S3 URI  : {SCRIPTS_S3_URI}")
print(f"Local pipeline source   : {LOCAL_PIPELINE_SRC}")


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Loaded MLflow App config from mlflow_app_config_team04_s403.json
MLflow App tags:
  Semester: 26S1
  sagemaker:domain-arn: arn:aws:sagemaker:ap-southeast-1:044528205969:domain/d-5q0cdlsfisve
  ProjectName: credit-card-fraud-detection
  sagemaker:space-arn: arn:aws:sagemaker:ap-southeast-1:044528205969:space/d-5q0cdlsfisve/team04-shared
  Course: ITI113
  TeamId: team04
  CreatedByNotebook: ModelB
  StudentId: s401
[OK] MLflow App tag TeamId=team04 matches notebook TEAM_ID=team04
Pipeline                : iti113-team04-credit-card-fraud-detection
Bucket                  : nyp-26s1-iti113
Team prefix             : iti113/team04/data/credit-card-fraud-detection
Semester                : 26S1
Region                  : ap-southeast-1
SageMaker role          : arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team04
MLflow App ARN          : arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-ANFQ3RACFV2G
MLflow experiment       : ITI113/team04/Experiment-01
Pipeline sour

In [2]:
### RESET FRAUDTEST_CICD (RUN MANUALLY BEFORE A FRESH CI/CD DEMO)
import boto3

s3 = boto3.client('s3')

s3.copy_object(
    Bucket=BUCKET,
    CopySource=f"{BUCKET}/{PREFIX}/raw/fraudTest.csv",
    Key=f"{PREFIX}/raw/fraudTest_cicd.csv"
)
print("fraudTest_cicd.csv reset to original 555k baseline.")

fraudTest_cicd.csv reset to original 555k baseline.


In [3]:
!aws s3 ls s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/raw/

2026-08-09 11:50:03  150354339 fraudTest.csv
2026-08-19 13:52:20  133518638 fraudTest1.csv
2026-08-19 13:57:48   14880548 fraudTest2.csv
2026-08-26 09:57:53  150354339 fraudTest_cicd.csv
2026-08-09 11:50:05  351238196 fraudTrain.csv
2026-08-09 14:01:54   53468778 fraudTrain_sample.csv


### **STEP 02: PRECHECK SAGEMAKER MLFLOW APP CONNECTION**

In [4]:
import mlflow
import time

mlflow.set_tracking_uri(MLFLOW_APP_ARN)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name=f"{TEAM_ID}_pipeline_notebook_precheck_{int(time.time())}") as run:
    mlflow.set_tags({
        "course": "ITI113",
        "semester": SEMESTER,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "dataset": "heart-disease",
        "run_type": "sagemaker_pipeline_precheck",
        "tracking_backend": "sagemaker_mlflow_app",
        "mlflow_app_arn": MLFLOW_APP_ARN,
    })
    mlflow.log_param("source", "notebook_03_precheck")
    mlflow.log_metric("connection_success", 1)

    precheck_run_id = run.info.run_id
    precheck_experiment_id = run.info.experiment_id

print("SageMaker MLflow App precheck completed.")
print("MLflow App ARN:", MLFLOW_APP_ARN)
print("Experiment:", MLFLOW_EXPERIMENT_NAME)
print("Experiment ID:", precheck_experiment_id)
print("Run ID:", precheck_run_id)
print("\nIgnore any generic mlflow.sagemaker.app.aws link printed by MLflow above.")
print("Use the presigned links below instead:")
print_mlflow_presigned_links(
    experiment_id=precheck_experiment_id,
    run_id=precheck_run_id
)


🏃 View run team04_pipeline_notebook_precheck_1787738315 at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/2/runs/549eea76b81c4d8d954f0d227d880b28
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/2
SageMaker MLflow App precheck completed.
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-ANFQ3RACFV2G
Experiment: ITI113/team04/Experiment-01
Experiment ID: 2
Run ID: 549eea76b81c4d8d954f0d227d880b28

Ignore any generic mlflow.sagemaker.app.aws link printed by MLflow above.
Use the presigned links below instead:


Presigned MLflow experiment URL:
https://app-ANFQ3RACFV2G.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IkNGS01STyIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNDlkYWVtTnplaWNkK1lnWlZvTWd5SCtBcWlmZXJnS3ZJOUcxbkFxK2JPMG9BWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGdVFsYzBNa2hsYURsYVlVZzRjRXRGVDJkc2JuZHlabTE0UjA4eU5EZ3ZNVXRpUlRVNWRYcGxkVkkyYW1vMFRXUXdUbnBsYTI5MlRqTlNlSEZpV2tvMFp6MDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFSdllrQVVrenlvVjBoWFJwZDVDMzQ4QUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF3eWV2VFRGZUZyc2J3OUZHUUNBUkNBTzRXQlVNRE9zZDFjYnNmM3BPTit0T3oxUjQvNlZUZDMrcU00YkdaRkJJRHk4NU5EU3lRUVo0RXNzSytabWJFNnV3K3B4eS9MT2J2N3crTllBZ0FBRUFDQ3o3OWovOVdKNy9sV0ppdmdTcmtRMDVFUklzOEQrSFNsSEV0aXlSM25LazJsTWZMbFFXTj


Presigned MLflow run URL:
https://app-ANFQ3RACFV2G.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IkJPRk00TCIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNGVSVUxqV1JMNDdKZnJkUGpyWitPYWtxMkFDNXd6M1RFSENkZVh4Vy9veDBBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGclVFUkdWVVI1YVRCWk1HNTJlVEU1TkRWbk9GSkZjMWRQVml0SGFVWjBjVGRtWnpCMlRXNUtVRVZHWTB0SWRFbDNibFZPZVVseVVGVjRVaTk2VDFFM1VUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFWQVMzSlhYbGRRdE9oSlQralFuY2NNQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF3bzFrNnd3R29FbEJOWHVhb0NBUkNBTzFiKzJtMGhrMGEvTXVWVHN5RE85TG5xNXhzUkh5UmhwQUdGRGxwekh1YWFRZEczNnorQytIbVFGTldBMWMwMkE4TTlpeXMxZU5CZ053ODFBZ0FBRUFDN0hQeE5jRlBXZGlhVklicmdPWW8wOTl3OVR6U05VYXJVZFVWSU0vZEd3UmQycEZqdmJlQ1lSUjJB

### **STEP 03: WRITE PIPELINE SCRIPTS**

In [5]:
os.makedirs('src', exist_ok=True)
print('src/ directory ready')


src/ directory ready


In [6]:
%%writefile src/requirements.txt
scikit-learn==1.7.2

Overwriting src/requirements.txt


In [7]:
%%writefile src/preprocess.py
"""SageMaker Processing Job — Model B (Vanitha, MLP track): feature engineering,
StandardScaler + OneHotEncoder fitting, train/valid/test split.
Ported from model developer B's team0401_Vanitha_Final_Model_B_Retrain_Pipeline_v1_4.ipynb
(Sections 2-5), so preprocessing produces exactly the fields the new training/inference
code expects.
NOTE: RAW_ENGINEERED_COLUMNS / CONTINUOUS_FEATURES / PASSTHROUGH_FEATURES are not sourced
from m03_feature_columns.json (not provided) -- they are inferred from the retrain notebook's
own code and the "13 raw engineered columns" count confirmed in its Section 3 Findings.
Confirm against the manifest with the model developer when available.
"""
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn==1.7.2"])

import argparse
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

RAW_ENGINEERED_COLUMNS = [
    "amt", "amt_log", "category", "city_pop", "age",
    "trans_hour_sin", "trans_hour_cos", "day_of_week", "is_weekend", "gender_binary",
    "txn_velocity_1h", "txn_velocity_6h", "txn_velocity_24h",
]
CONTINUOUS_FEATURES = [
    "amt_log", "age", "city_pop", "txn_velocity_1h", "txn_velocity_6h", "txn_velocity_24h",
]
PASSTHROUGH_FEATURES = ["amt", "trans_hour_sin", "trans_hour_cos", "day_of_week", "is_weekend", "gender_binary"]
VELOCITY_WINDOWS = {"txn_velocity_1h": "1h", "txn_velocity_6h": "6h", "txn_velocity_24h": "24h"}

def add_base_features(data):
    x = data.copy()
    x["trans_date_trans_time"] = pd.to_datetime(x["trans_date_trans_time"], errors="coerce")
    x["dob"] = pd.to_datetime(x["dob"], errors="coerce")

    x["age"] = x["trans_date_trans_time"].dt.year - x["dob"].dt.year
    before_birthday = (
        (x["trans_date_trans_time"].dt.month < x["dob"].dt.month)
        | (
            (x["trans_date_trans_time"].dt.month == x["dob"].dt.month)
            & (x["trans_date_trans_time"].dt.day < x["dob"].dt.day)
        )
    )
    x["age"] -= before_birthday.astype(int)

    x["amt_log"] = np.log1p(x["amt"].clip(lower=0))

    hour = x["trans_date_trans_time"].dt.hour
    x["trans_hour_sin"] = np.sin(2 * np.pi * hour / 24)
    x["trans_hour_cos"] = np.cos(2 * np.pi * hour / 24)

    x["day_of_week"] = x["trans_date_trans_time"].dt.dayofweek
    x["is_weekend"] = (x["day_of_week"] >= 5).astype(int)

    x["gender_binary"] = x["gender"].map({"F": 0, "M": 1})
    return x

def add_velocity_features(data, windows):
    x = data.copy()
    x_sorted = x.sort_values(["cc_num", "trans_date_trans_time"]).set_index("trans_date_trans_time")
    for col_name, window in windows.items():
        counts = (
            x_sorted.groupby("cc_num")["amt"]
            .rolling(window, closed="left")
            .count()
            .fillna(0)
        )
        x_sorted[col_name] = counts.values
    x_sorted = x_sorted.reset_index()
    velocity_cols = list(windows.keys())
    return x.merge(x_sorted[["trans_num"] + velocity_cols], on="trans_num", how="left")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--test-size", type=float, default=0.20)
    parser.add_argument("--valid-size", type=float, default=0.25)
    parser.add_argument("--random-state", type=int, default=42)
    args = parser.parse_args()

    base_dir = "/opt/ml/processing"
    import glob
    df = pd.read_csv(glob.glob(f"{base_dir}/input/*.csv")[0])

    df = add_base_features(df)
    df = add_velocity_features(df, VELOCITY_WINDOWS)

    keep_cols = RAW_ENGINEERED_COLUMNS + ["is_fraud"]
    model_df = (
        df[keep_cols]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .reset_index(drop=True)
    )

    X = model_df[RAW_ENGINEERED_COLUMNS]
    y = model_df["is_fraud"].astype(int)

    X_dev, X_test, y_dev, y_test = train_test_split(
        X, y, test_size=args.test_size, stratify=y, random_state=args.random_state
    )
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_dev, y_dev, test_size=args.valid_size, stratify=y_dev, random_state=args.random_state
    )

    scaler = StandardScaler()
    scaler.fit(X_train[CONTINUOUS_FEATURES])

    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    ohe.fit(X_train[["category"]])
    ohe_columns = [f"category_{c}" for c in ohe.categories_[0]]

    out_dir = Path(base_dir) / "output"
    for name, Xdf, ydf in [("train", X_train, y_train), ("valid", X_valid, y_valid), ("test", X_test, y_test)]:
        Xdf[RAW_ENGINEERED_COLUMNS].assign(is_fraud=ydf.values).to_csv(out_dir / f"{name}.csv", index=False)

    preprocessing_bundle = {
        "scaler": scaler,
        "ohe": ohe,
        "continuous_features": CONTINUOUS_FEATURES,
        "ohe_columns": ohe_columns,
        "passthrough_features": PASSTHROUGH_FEATURES,
        "raw_engineered_columns": RAW_ENGINEERED_COLUMNS,
    }
    joblib.dump(preprocessing_bundle, out_dir / "modelb_preprocessing_bundle.joblib")

    print("Preprocessing complete.")
    print(f"Train: {len(X_train)} | Valid: {len(X_valid)} | Test: {len(X_test)}")
    print(f"Category levels: {len(ohe_columns)} -> {ohe_columns}")

Overwriting src/preprocess.py


In [8]:
%%writefile src/feature_transformer.py
"""Shared feature-transform class for Model B (MLP track).
Defined in its own module — not inside train.py or inference.py — so that when the
fitted sklearn.Pipeline is saved with joblib/pickle in train.py and reloaded in
inference.py, Python resolves this class from the same module path
(feature_transformer.M03FeatureTransformer) in both places. If this class were
defined inline in train.py instead, pickle would record it as belonging to
train.py's own run context, and inference.py would fail to unpickle it.
Copied verbatim (transform logic) from model developer B's
team0401_Vanitha_Final_Model_B_Retrain_Pipeline_v1_4.ipynb, Section 5.
"""

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class M03FeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, scaler, ohe, continuous_features, ohe_columns, passthrough_features):
        self.scaler = scaler
        self.ohe = ohe
        self.continuous_features = continuous_features
        self.ohe_columns = ohe_columns
        self.passthrough_features = passthrough_features

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        scaled = pd.DataFrame(
            self.scaler.transform(X[self.continuous_features]),
            columns=self.continuous_features,
            index=X.index,
        )
        encoded = pd.DataFrame(
            self.ohe.transform(X[['category']]),
            columns=self.ohe_columns,
            index=X.index,
        )
        passthrough = X[self.passthrough_features].reset_index(drop=True)
        result = pd.concat(
            [scaled.reset_index(drop=True), encoded.reset_index(drop=True), passthrough],
            axis=1,
        )
        result.index = X.index
        return result

    def get_feature_names_out(self, input_features=None):
        return np.array(self.continuous_features + self.ohe_columns + self.passthrough_features)

Overwriting src/feature_transformer.py


In [9]:
%%writefile src/train.py
"""SageMaker Training Job — Model B (Vanitha, MLP track): trains MLPClassifier wrapped in a
feature-transform Pipeline, threshold selection, artifact export.
Hyperparameters are Vanitha's locked final MLP config (team0401_Vanitha_Final_Model_B_Retrain_Pipeline_v1_4.ipynb,
Section 6) -- a snapshot from her hyperparameter search, not meant to be re-tuned per pipeline
run, so hardcoded here rather than exposed as CLI arguments (matching her own retrain notebook).
The classifier__sample_weight line below is copied FAITHFULLY from her Section 6 code, unchanged,
even though scikit-learn's MLPClassifier.fit() does not accept a sample_weight argument -- if this
raises a TypeError when run, that is evidence her own code does not execute as written, not
something altered on the MLOps side. See conversation notes before "fixing" this.
Saves one artifact: modelb_pipeline.joblib (feature_transformer.M03FeatureTransformer + MLPClassifier,
as one fitted sklearn.Pipeline) plus modelb_deployment_contract.json.
"""
import argparse
import json
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score,
    recall_score, f1_score, confusion_matrix, precision_recall_curve
)
from feature_transformer import M03FeatureTransformer

# Locked final MLP hyperparameters (Final Model B, Section 4a) -- snapshot, not re-tuned here.
FINAL_MLP_PARAMS = {
    "hidden_layer_sizes": (128, 64),
    "activation": "relu",
    "solver": "adam",
    "alpha": 0.01,
    "learning_rate_init": 0.001,
    "max_iter": 200,
    "early_stopping": True,
    "n_iter_no_change": 10,
    "validation_fraction": 0.1,
}

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--random-state", type=int, default=42)
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--valid", type=str, default=os.environ.get("SM_CHANNEL_VALID"))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST"))
    parser.add_argument("--preprocessor", type=str, default=os.environ.get("SM_CHANNEL_PREPROCESSOR"))
    parser.add_argument("--sm-model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    args = parser.parse_args()

    train_df = pd.read_csv(os.path.join(args.train, "train.csv"))
    valid_df = pd.read_csv(os.path.join(args.valid, "valid.csv"))
    test_df = pd.read_csv(os.path.join(args.test, "test.csv"))

    bundle = joblib.load(os.path.join(args.preprocessor, "modelb_preprocessing_bundle.joblib"))
    raw_cols = bundle["raw_engineered_columns"]

    X_train, y_train = train_df[raw_cols], train_df["is_fraud"].astype(int)
    X_valid, y_valid = valid_df[raw_cols], valid_df["is_fraud"].astype(int)
    X_test, y_test = test_df[raw_cols], test_df["is_fraud"].astype(int)

    feature_transformer = M03FeatureTransformer(
        scaler=bundle["scaler"], ohe=bundle["ohe"],
        continuous_features=bundle["continuous_features"],
        ohe_columns=bundle["ohe_columns"],
        passthrough_features=bundle["passthrough_features"],
    )

    model = Pipeline([
        ("features", feature_transformer),
        ("classifier", MLPClassifier(random_state=args.random_state, **FINAL_MLP_PARAMS)),
    ])

    # Faithful to her Section 6 code -- see module docstring above.
    sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
    model.fit(X_train, y_train, classifier__sample_weight=sample_weight)

    # Threshold selection on validation -- fine-grained scan over every threshold
    # precision_recall_curve() produces, matching her established standard
    # (team0401_Vanitha_Final_Model_B_Retrain_Pipeline_v1_4.ipynb Section 7).
    valid_prob = model.predict_proba(X_valid)[:, 1]
    precisions, recalls, pr_thresholds = precision_recall_curve(y_valid, valid_prob)
    precisions, recalls = precisions[:-1], recalls[:-1]
    f1_scores = np.where(
        (precisions + recalls) > 0,
        2 * precisions * recalls / (precisions + recalls),
        0.0,
    )
    best_idx = int(np.argmax(f1_scores))
    best_threshold = float(pr_thresholds[best_idx])

    test_prob = model.predict_proba(X_test)[:, 1]
    test_pred = (test_prob >= best_threshold).astype(int)

    test_roc_auc = roc_auc_score(y_test, test_prob)
    test_pr_auc = average_precision_score(y_test, test_prob)
    test_precision = precision_score(y_test, test_pred, zero_division=0)
    test_recall = recall_score(y_test, test_pred, zero_division=0)
    test_f1 = f1_score(y_test, test_pred, zero_division=0)

    cm = confusion_matrix(y_test, test_pred)
    tn, fp, fn, tp = cm.ravel()
    test_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    test_fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    # NOTE: print format below must stay exact -- the pipeline's TrainingStep
    # metric_definitions regex (STEP 06, "STEP 2: TRAININGSTEP" cell) parses these lines.
    print(f"Test ROC-AUC: {test_roc_auc:.4f}")
    print(f"Test PR-AUC: {test_pr_auc:.4f}")
    print(f"Test Precision: {test_precision:.4f}")
    print(f"Test Recall: {test_recall:.4f}")
    print(f"Test F1: {test_f1:.4f}")
    print(f"Test FPR: {test_fpr:.6f}")
    print(f"Test FNR: {test_fnr:.6f}")
    print(f"Selected Threshold: {best_threshold:.4f}")

    # --- Save artifacts ---
    joblib.dump(model, os.path.join(args.sm_model_dir, "modelb_pipeline.joblib"))

    # Also save a version-agnostic weights bundle (plain numpy .npz, not pickle)
    # for inference.py to use instead of unpickling this sklearn Pipeline.
    # See inference.py docstring for why: the serving container we're forced
    # to use cannot run scikit-learn>=1.7, so inference reimplements the
    # transform + MLP forward pass by hand from these raw numbers.
    fitted_transformer = model.named_steps["features"]
    mlp = model.named_steps["classifier"]

    weights = {
        "scaler_mean": fitted_transformer.scaler.mean_,
        "scaler_scale": fitted_transformer.scaler.scale_,
    }
    for idx, (w, b) in enumerate(zip(mlp.coefs_, mlp.intercepts_)):
        weights[f"coef_{idx}"] = w
        weights[f"intercept_{idx}"] = b
    np.savez(os.path.join(args.sm_model_dir, "modelb_weights.npz"), **weights)

    deployment_contract = {
        "raw_engineered_columns": raw_cols,
        "continuous_features": bundle["continuous_features"],
        "ohe_categories": bundle["ohe"].categories_[0].tolist(),
        "ohe_columns": bundle["ohe_columns"],
        "passthrough_features": bundle["passthrough_features"],
        "n_mlp_layers": len(mlp.coefs_),
        "operating_threshold": best_threshold,
        "decision_rule": "fraud if predicted_probability >= operating_threshold",
    }
    with open(os.path.join(args.sm_model_dir, "modelb_deployment_contract.json"), "w") as f:
        json.dump(deployment_contract, f, indent=2)

    print("Model artifacts saved: modelb_pipeline.joblib, modelb_weights.npz, modelb_deployment_contract.json")

Overwriting src/train.py


In [10]:
%%writefile src/inference.py
"""SageMaker inference handler for Model B (MLP) -- pure numpy/pandas implementation.

Deliberately avoids scikit-learn and joblib entirely at serve time. Two hard
constraints forced this design:
  1. The scikit-learn "1.4-2" framework container cannot host real-time
     entry-point inference as a non-root user (a container-level bug,
     confirmed via CloudWatch logs across five independent fix attempts --
     see conversation notes). We must deploy on the older, root-based
     "1.2-1" container instead.
  2. "1.2-1" runs Python 3.9, but scikit-learn>=1.7 (needed to unpickle a
     Pipeline fitted under it) requires Python>=3.10 -- so scikit-learn
     itself cannot even be installed there.

So instead of unpickling a fitted sklearn Pipeline, this handler loads the
model's raw learned numbers (StandardScaler mean/scale, MLPClassifier
weights) from a plain numpy .npz file -- a format that is stable across
numpy versions, unlike pickle -- and reimplements preprocessing + the MLP
forward pass by hand. This is mathematically identical to what the fitted
sklearn Pipeline computes (same scaler formula, same weight matrices, same
relu/sigmoid math) -- it does not change the model or its learned behavior
in any way, only how the already-trained numbers are evaluated at serve time.

Expects model_dir to contain: modelb_weights.npz, modelb_deployment_contract.json
(both produced by train.py).

IMPORTANT -- interim design decision (velocity-at-inference deferred):
Same as before: callers must submit the already-engineered fields (the same
columns preprocess.py writes to train.csv/valid.csv/test.csv, minus
is_fraud). Velocity-at-inference for live single transactions is not yet
implemented.
"""
import json
import os
import numpy as np
import pandas as pd


def _relu(z):
    return np.maximum(0, z)


def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def model_fn(model_dir):
    """Load the model's raw weights and contract once, at container startup."""
    weights = np.load(os.path.join(model_dir, "modelb_weights.npz"))
    with open(os.path.join(model_dir, "modelb_deployment_contract.json"), "r", encoding="utf-8") as f:
        contract = json.load(f)

    n_layers = contract["n_mlp_layers"]
    coefs = [weights[f"coef_{i}"] for i in range(n_layers)]
    intercepts = [weights[f"intercept_{i}"] for i in range(n_layers)]

    return {
        "scaler_mean": weights["scaler_mean"],
        "scaler_scale": weights["scaler_scale"],
        "coefs": coefs,
        "intercepts": intercepts,
        "contract": contract,
        "operating_threshold": float(contract["operating_threshold"]),
    }


def input_fn(request_body, request_content_type):
    """Parse the incoming request into a DataFrame of one or more transactions."""
    if request_content_type != "application/json":
        raise ValueError(
            f"Unsupported content type: {request_content_type}. Expected application/json."
        )
    payload = json.loads(request_body)
    records = payload if isinstance(payload, list) else [payload]
    return pd.DataFrame.from_records(records)


def _transform(input_data, model_bundle):
    """Reimplements M03FeatureTransformer.transform() using plain numpy/pandas
    (no sklearn StandardScaler/OneHotEncoder objects involved)."""
    contract = model_bundle["contract"]
    continuous_features = contract["continuous_features"]
    ohe_categories = contract["ohe_categories"]
    passthrough_features = contract["passthrough_features"]

    required = continuous_features + ["category"] + passthrough_features
    missing = [c for c in required if c not in input_data.columns]
    if missing:
        raise ValueError(
            "Input is missing required pre-engineered fields "
            f"(velocity-at-inference is not yet implemented, so raw fields "
            f"are not accepted): {missing}"
        )

    # StandardScaler equivalent: (x - mean) / scale
    continuous_values = input_data[continuous_features].to_numpy(dtype=float)
    scaled = (continuous_values - model_bundle["scaler_mean"]) / model_bundle["scaler_scale"]

    # OneHotEncoder(handle_unknown="ignore") equivalent: all-zero row for unseen categories
    category_values = input_data["category"].to_numpy()
    encoded = np.zeros((len(input_data), len(ohe_categories)), dtype=float)
    category_index = {cat: idx for idx, cat in enumerate(ohe_categories)}
    for row_idx, cat in enumerate(category_values):
        col_idx = category_index.get(cat)
        if col_idx is not None:
            encoded[row_idx, col_idx] = 1.0

    passthrough_values = input_data[passthrough_features].to_numpy(dtype=float)

    return np.concatenate([scaled, encoded, passthrough_values], axis=1)


def predict_fn(input_data, model_bundle):
    """Reimplements Pipeline.predict_proba() (features transform + MLP forward pass)."""
    X = _transform(input_data, model_bundle)

    activations = X
    coefs = model_bundle["coefs"]
    intercepts = model_bundle["intercepts"]
    for i in range(len(coefs) - 1):
        activations = _relu(activations @ coefs[i] + intercepts[i])
    logits = activations @ coefs[-1] + intercepts[-1]
    probability = _sigmoid(logits[:, 0])

    threshold = model_bundle["operating_threshold"]
    prediction = (probability >= threshold).astype(int)
    return pd.DataFrame({
        "fraud_probability": probability,
        "is_fraud_predicted": prediction,
        "operating_threshold": threshold,
    })


def output_fn(prediction, response_content_type):
    """Serialise the prediction DataFrame back to the response payload."""
    if response_content_type != "application/json":
        raise ValueError(
            f"Unsupported accept type: {response_content_type}. Expected application/json."
        )
    return json.dumps(prediction.to_dict(orient="records")), response_content_type

Overwriting src/inference.py


In [11]:
# CONFIRM PREPROCESS.PY, TRAIN.PY, INFERENCE.PY WERE WRITTEN LOCALLY (PRINT FILE SIZES)
# No MLflow requirements file is needed in the SageMaker training container.
# MLflow logging is done after the pipeline completes, from this notebook.
# The training script now saves a joblib deployment bundle containing:
#   - trained model
#   - fitted scaler
#   - missing-value fill values
#   - raw and processed feature column order

print("Scripts written:")
for fn in ["preprocess.py", "train.py", "inference.py"]:
    size = os.path.getsize(f"src/{fn}")
    print(f"  src/{fn}  ({size} bytes)")

Scripts written:
  src/preprocess.py  (5276 bytes)
  src/train.py  (7166 bytes)
  src/inference.py  (5855 bytes)


### **STEP 04: UPLOAD PIPELINE SOURCE FILES TO S3**

In [12]:
from pathlib import Path

s3_client = boto3.client("s3")

SOURCE_DIR = Path("src")
FILES_TO_UPLOAD = [
    "preprocess.py",
    "train.py",
    "inference.py",
    "feature_transformer.py",
    "requirements.txt",
]

for filename in FILES_TO_UPLOAD:
    local_path = SOURCE_DIR / filename
    s3_key = f"{SCRIPTS_S3_PREFIX}/{filename}"

    if not local_path.exists():
        raise FileNotFoundError(f"Missing local source file: {local_path}")

    s3_client.upload_file(str(local_path), BUCKET, s3_key)
    print(f"Uploaded {local_path} -> s3://{BUCKET}/{s3_key}")

print("Pipeline source files uploaded to:", SCRIPTS_S3_URI)

Uploaded src/preprocess.py -> s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/pipeline_src/preprocess.py
Uploaded src/train.py -> s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/pipeline_src/train.py
Uploaded src/inference.py -> s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/pipeline_src/inference.py
Uploaded src/feature_transformer.py -> s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/pipeline_src/feature_transformer.py
Uploaded src/requirements.txt -> s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/pipeline_src/requirements.txt
Pipeline source files uploaded to: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/pipeline_src


### **STEP 05: DOWNLOAD PIPELINE SOURCE FILES FROM S3**

In [13]:
import shutil

local_src = Path(LOCAL_PIPELINE_SRC)

if local_src.exists():
    shutil.rmtree(local_src)

local_src.mkdir(parents=True, exist_ok=True)

for filename in FILES_TO_UPLOAD:
    s3_key = f"{SCRIPTS_S3_PREFIX}/{filename}"
    local_path = local_src / filename

    s3_client.download_file(BUCKET, s3_key, str(local_path))
    print(f"Downloaded s3://{BUCKET}/{s3_key} -> {local_path}")

print("Downloaded files:")
for p in sorted(local_src.iterdir()):
    print("-", p)

Downloaded s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/pipeline_src/preprocess.py -> pipeline_src/preprocess.py
Downloaded s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/pipeline_src/train.py -> pipeline_src/train.py
Downloaded s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/pipeline_src/inference.py -> pipeline_src/inference.py
Downloaded s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/pipeline_src/feature_transformer.py -> pipeline_src/feature_transformer.py
Downloaded s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/pipeline_src/requirements.txt -> pipeline_src/requirements.txt
Downloaded files:
- pipeline_src/feature_transformer.py
- pipeline_src/inference.py
- pipeline_src/preprocess.py
- pipeline_src/requirements.txt
- pipeline_src/train.py


In [14]:
LOCAL_INFERENCE_SRC = "pipeline_src_inference"

local_inference_src = Path(LOCAL_INFERENCE_SRC)
if local_inference_src.exists():
    shutil.rmtree(local_inference_src)
local_inference_src.mkdir(parents=True, exist_ok=True)

INFERENCE_ONLY_FILES = ["inference.py", "feature_transformer.py"]

for filename in INFERENCE_ONLY_FILES:
    src_path = local_src / filename
    dst_path = local_inference_src / filename
    shutil.copy(src_path, dst_path)
    print(f"Copied {src_path} -> {dst_path}")

print("Inference-only source folder (no requirements.txt):")
for p in sorted(local_inference_src.iterdir()):
    print("-", p)

Copied pipeline_src/inference.py -> pipeline_src_inference/inference.py
Copied pipeline_src/feature_transformer.py -> pipeline_src_inference/feature_transformer.py
Inference-only source folder (no requirements.txt):
- pipeline_src_inference/feature_transformer.py
- pipeline_src_inference/inference.py


### **STEP 06: DEFINE THE SAGEMAKER PIPELINE**

In [15]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.parameters import ParameterFloat, ParameterInteger
from sagemaker.workflow.model_step import ModelStep
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.model import Model
from sagemaker.workflow.pipeline_context import PipelineSession

pipeline_session = PipelineSession()

# Pipeline parameters — can be overridden at execution time
p_n_est    = ParameterInteger(name='NEstimators',    default_value=200)
p_depth    = ParameterInteger(name='MaxDepth',       default_value=6)

# p_samples  = ParameterInteger(name='MinSamplesLeaf', default_value=4)
p_lr          = ParameterFloat(  name='LearningRate',     default_value=0.05)
p_subsample   = ParameterFloat(  name='Subsample',        default_value=0.8)
p_colsample   = ParameterFloat(  name='ColsampleBytree',  default_value=0.8)

p_gate     = ParameterFloat(  name='QualityGateAUC', default_value=QUALITY_GATE_AUC)

print('Pipeline parameters defined.')

Pipeline parameters defined.


In [16]:
# STEP 1: PROCESSINGSTEP

processor = SKLearnProcessor(
    framework_version='1.4-2', instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1, role=role, sagemaker_session=pipeline_session,
    base_job_name=f'iti113-{TEAM_ID}-{STUDENT_ID}-process')

step_process = ProcessingStep(
    name='Team04-PreprocessData',
    processor=processor,
    inputs=[ProcessingInput(source=RAW_DATA_URI,
                            destination='/opt/ml/processing/input')],
    outputs=[ProcessingOutput(output_name='processed',
                              source='/opt/ml/processing/output',
                              destination=f'{PIPELINE_ROOT}/processed')],
    code=f'{LOCAL_PIPELINE_SRC}/preprocess.py',
    job_arguments=['--test-size','0.2','--random-state','42']
)
print('Step 1 (ProcessingStep) defined.')


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


/opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:138: SageMakerV2DeprecationWarning: SKLearnProcessor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `DataProcessor` (`from sagemaker.mlops.processing import DataProcessor`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Step 1 (ProcessingStep) defined.


In [17]:
# STEP 2: TRAININGSTEP
# The training container receives no Databricks host, token, or MLflow dependency.
# It only trains the model and prints metrics for SageMaker to capture.

estimator = SKLearn(
    entry_point="train.py",
    source_dir=LOCAL_PIPELINE_SRC,
    dependencies=[f'{LOCAL_PIPELINE_SRC}/requirements.txt'],
    framework_version="1.4-2",
    instance_type=TRAINING_INSTANCE_TYPE,
    instance_count=1,
    role=role,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-train",
    sagemaker_session=pipeline_session,
    hyperparameters={
        "random-state": 42,
    },
    environment={
        "TEAM_ID": TEAM_ID,
        "STUDENT_ID": STUDENT_ID,
        "SEMESTER": SEMESTER,
    },

    metric_definitions=[
        {"Name": "test_auc_roc", "Regex": "Test ROC-AUC: ([0-9\\.]+)"},
        {"Name": "test_pr_auc", "Regex": "Test PR-AUC: ([0-9\\.]+)"},
        {"Name": "test_precision", "Regex": "Test Precision: ([0-9\\.]+)"},
        {"Name": "test_recall", "Regex": "Test Recall: ([0-9\\.]+)"},
        {"Name": "test_f1", "Regex": "Test F1: ([0-9\\.]+)"},
        {"Name": "test_fpr", "Regex": "Test FPR: ([0-9\\.]+)"},
        {"Name": "test_fnr", "Regex": "Test FNR: ([0-9\\.]+)"},
],

    tags=[
        {"Key": "Course", "Value": "ITI113"},
        {"Key": "Semester", "Value": SEMESTER},
        {"Key": "Team", "Value": TEAM_ID},
        {"Key": "Student", "Value": STUDENT_ID},
    ],
)

processed_uri = step_process.properties.ProcessingOutputConfig.Outputs["processed"].S3Output.S3Uri

step_train = TrainingStep(
    name="Team04-TrainModel",
    estimator=estimator,
    inputs={
        "train": sagemaker.inputs.TrainingInput(
            s3_data=processed_uri,
            content_type="text/csv"
        ),
        "test": sagemaker.inputs.TrainingInput(
            s3_data=processed_uri,
            content_type="text/csv"
        ),

        "valid": sagemaker.inputs.TrainingInput(
            s3_data=processed_uri,
            content_type="text/csv"
        ),

        "preprocessor": sagemaker.inputs.TrainingInput(
            s3_data=processed_uri,
            content_type="application/octet-stream"
        ),
    },
)

print("Step 2 (TrainingStep) defined.")

/opt/conda/lib/python3.12/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: SKLearn is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Step 2 (TrainingStep) defined.


In [18]:
# STEP 3: MODELSTEP — REGISTER IN SAGEMAKER MODEL REGISTRY

inference_image_uri = sagemaker.image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.2-1",
    instance_type=PROCESSING_INSTANCE_TYPE,
)

model = Model(
    image_uri=inference_image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role,
    entry_point='inference.py',
    source_dir=LOCAL_INFERENCE_SRC
)
step_register = ModelStep(
    name='Team04-RegisterModel',
    step_args=model.register(
        content_types=['application/json'],
        response_types=['application/json'],
        inference_instances=['ml.m5.large'],
        transform_instances=['ml.m5.large'],
        model_package_group_name=MODEL_PACKAGE_GROUP,
        approval_status='PendingManualApproval',
    )
)
print('Step 3 (ModelStep) defined.')


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


/opt/conda/lib/python3.12/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: Model is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


Step 3 (ModelStep) defined.


In [19]:
# STEP 4: CONDITIONSTEP — GATE ON SAGEMAKER-CAPTURED TEST AUC

# The training script prints:
#     Test AUC-ROC: 0.xxxx
# and the estimator metric_definitions capture this as "test_auc_roc".
# This avoids relying on Databricks Model Registry or a separate evaluation file.
condition = ConditionGreaterThanOrEqualTo(
    left=step_train.properties.FinalMetricDataList["test_pr_auc"].Value,
    right=p_gate
)

# step_register need to be declared first before used in step_condition
# and that is why register cell comes before condition cell
# NOTE: step_register (defined above) is only referenced here as a Python
# object — it does NOT execute until AFTER this condition passes.
# Cell order != execution order. Actual order:
#   PreprocessData -> TrainModel -> AUCQualityGate -> RegisterModel

step_condition = ConditionStep(
    name="Team04-AUCQualityGate",
    conditions=[condition],
    if_steps=[step_register],
    else_steps=[]
)
print("Step 4 (ConditionStep) defined.")


Step 4 (ConditionStep) defined.


In [20]:
# STEP 5: ASSEMBLE AND UPSERT THE PIPELINE

pipeline = Pipeline(
    name=PIPELINE_NAME,

    parameters=[p_gate],
    
    steps=[step_process, step_train, step_condition], # step_register nested inside step_condition
    sagemaker_session=pipeline_session
)
pipeline.upsert(role_arn=role)
print(f'Pipeline "{PIPELINE_NAME}" upserted.')
print('View in SageMaker Studio: left sidebar -> Pipelines')

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:119: SageMakerV2DeprecationWarning: Pipeline is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `Pipeline` (`from sagemaker.mlops.pipeline import Pipeline`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Pipeline "iti113-team04-credit-card-fraud-detection" upserted.
View in SageMaker Studio: left sidebar -> Pipelines
